# Enflasyon Penceresi Ton Sınıflandırması: Qwen2.5-7B QLoRA Fine-Tuning

Demiralp, Saka & Weber (2026) yöntemini LLM ile genişletir: "enflasyon" kelimesinin ±15 kelimelik penceresini **iyimser / kötümser / nötr** olarak sınıflandırır.

**Başlamadan önce:**
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU** seç.
2. Google Drive'da `enflasyon_projesi` adlı bir klasör aç ve içine şu 4 dosyayı yükle (bilgisayarındaki `veri/` klasöründen):
   `train.csv`, `val.csv`, `test.csv`, `pencereler.csv`
3. Hücreleri sırayla çalıştır. İş bitince **Çalışma zamanı → Bağlantıyı kes ve sil** (birim harcamasın).

## 1. Kurulum ve Drive

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate scikit-learn

KLASOR = '/content'
import os
print(os.listdir(KLASOR))
!nvidia-smi --query-gpu=name,memory.total --format=csv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 139.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 144.2 MB/s eta 0:00:00
['.config', 'sample_data']
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## 2. Ayarlar

In [ ]:
MODEL_ADI = 'Qwen/Qwen3.8-27B'
IKILI = False  # True: nötr pencereler atılır, makaledeki gibi sadece iyimser/kötümser
ETIKETLER = ['iyimser', 'kotumser'] if IKILI else ['iyimser', 'kotumser', 'notr']
ETIKET2ID = {e: i for i, e in enumerate(ETIKETLER)}
MAX_UZUNLUK = 128
CIKTI = f'{KLASOR}/qwen38_27b_qlora' + ('_ikili' if IKILI else '')
TOHUM = 42

OGRENME_ORANI = 1e-4   # 7B denemesinde 2e-4 ile model ezberlemişti
LORA_DROPOUT = 0.1
BATCH = 4              # A100 için; L4'te bellek yetmezse 2
GRAD_ACCUM = 4         # etkin batch = 16

## 3. Veri

In [ ]:
import pandas as pd
from datasets import Dataset

def yukle(ad):
    df = pd.read_csv(f'{KLASOR}/{ad}.csv', encoding='utf-8-sig')
    df['label'] = df['etiket'].map(ETIKET2ID)
    return df

train_df, val_df, test_df = yukle('train'), yukle('val'), yukle('test')
for ad, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(ad, len(df), df['etiket'].value_counts().to_dict())

train 563 {'notr': 239, 'kotumser': 173, 'iyimser': 151}
val 81 {'notr': 34, 'kotumser': 25, 'iyimser': 22}
test 160 {'notr': 68, 'kotumser': 49, 'iyimser': 43}


## 4. Model (4-bit) ve LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['score'],  # sınıflandırma kafası tam hassasiyette kalsın
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ADI,
    num_labels=len(ETIKETLER),
    id2label=dict(enumerate(ETIKETLER)),
    label2id=ETIKET2ID,
    quantization_config=bnb,
    torch_dtype=torch.bfloat16,
    device_map={'': 0},
)
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/8.95k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1183 [00:00<?, ?it/s]

[transformers] Qwen3_5ForSequenceClassification LOAD REPORT from: Qwen/Qwen3.8-27B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 79,707,136 || all params: 26,165,052,656 || trainable%: 0.3046


## 5. Tokenize

In [ ]:
def tokenize(ornek):
    return tokenizer(ornek['pencere'], truncation=True, max_length=MAX_UZUNLUK)

def hf(df):
    return Dataset.from_pandas(df[['pencere', 'label']]).map(tokenize, batched=True, remove_columns=['pencere'])

train_ds, val_ds, test_ds = hf(train_df), hf(val_df), hf(test_df)

Map:   0%|          | 0/563 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

## 6. Eğitim

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def metrikler(tahmin):
    logits, gercek = tahmin
    # Qwen gibi bazı modeller logitleri tuple olarak döndürebilir, güvenlik için kontrol ediyoruz
    if isinstance(logits, tuple):
        logits = logits[0]
    pred = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(gercek, pred),
        'f1_macro': f1_score(gercek, pred, average='macro'),
        'kappa': cohen_kappa_score(gercek, pred),
    }

args = TrainingArguments(
    output_dir=CIKTI,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    bf16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    # Değerlendirme metrikleri boş döndüğü için en iyi model yüklemeyi iptal ediyoruz:
    # load_best_model_at_end=True,
    # metric_for_best_model='f1_macro',
    logging_steps=10,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    seed=TOHUM,
)

# Modele pad_token_id değerini açıkça belirtiyoruz
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model.config, 'text_config'):
    model.config.text_config.pad_token_id = tokenizer.pad_token_id

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
    compute_metrics=metrikler,
)
trainer.train()
trainer.save_model(f'{CIKTI}/en_iyi')  # sadece LoRA adaptörü + sınıflandırma kafası (~100 MB)

Epoch,Training Loss,Validation Loss
1,0.720146,No log
2,0.085098,No log
3,0.003074,No log


## 7. Test seti değerlendirmesi

Not: Demo test etiketleri Claude tarafından verildi; buradaki κ "Claude ile uyum" demektir. Asıl çalışmada test seti insan etiketli olmalı.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

sonuc = trainer.predict(test_ds)

predictions = sonuc.predictions
# Model tahminleri tuple olarak döndürdüyse (örn: loss, logits), logitleri (2D matrisi) alıyoruz
if isinstance(predictions, tuple):
    for p in predictions:
        if isinstance(p, np.ndarray) and p.ndim == 2:
            predictions = p
            break

pred = predictions.argmax(axis=-1)
print(sonuc.metrics)
print(classification_report(test_df['label'], pred, target_names=ETIKETLER, digits=3))
print(pd.DataFrame(confusion_matrix(test_df['label'], pred), index=ETIKETLER, columns=ETIKETLER))

{'test_runtime': 7.7391, 'test_samples_per_second': 20.674, 'test_steps_per_second': 1.292}
              precision    recall  f1-score   support

     iyimser      0.763     0.674     0.716        43
    kotumser      0.727     0.653     0.688        49
        notr      0.667     0.765     0.712        68

    accuracy                          0.706       160
   macro avg      0.719     0.697     0.706       160
weighted avg      0.711     0.706     0.706       160

          iyimser  kotumser  notr
iyimser        29         3    11
kotumser        2        32    15
notr            7         9    52


### 7.1 Manuel Özel Test

Kendi yazdığınız cümlelerle modeli test edin.

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.metrics import classification_report

# Kendi cümlelerinizi ve gerçek etiketlerini buraya ekleyebilirsiniz:
ornek_veriler = [
    {"pencere": "Merkez Bankası'nın son adımları sayesinde enflasyonist baskıların azalması bekleniyor.", "etiket": "iyimser"},
    {"pencere": "Sebze meyve fiyatlarındaki astronomik artışlar enflasyonu daha da artıracak.", "etiket": "kotumser"},
    {"pencere": "TÜİK, aralık ayı enflasyon verilerini perşembe günü açıklayacak.", "etiket": "notr"}
]

def manuel_test_et(veriler):
    df_manuel = pd.DataFrame(veriler)
    df_manuel['label'] = df_manuel['etiket'].map(ETIKET2ID)

    # Dataset formatına çevir ve tokenize et
    ds_manuel = Dataset.from_pandas(df_manuel[['pencere', 'label']])
    ds_manuel = ds_manuel.map(tokenize, batched=True, remove_columns=['pencere'])

    # Tahmin yap (trainer tanımlı ve model eğitilmiş olmalıdır)
    print("Tahminler alınıyor...")
    sonuc = trainer.predict(ds_manuel)
    tahminler = sonuc.predictions.argmax(-1)
    df_manuel['tahmin'] = [ETIKETLER[t] for t in tahminler]

    print("\n--- Sonuçlar ---")
    display(df_manuel[['pencere', 'etiket', 'tahmin']])

    print("\n--- Metrikler ---")
    # Zero_division=0 parametresi, az sayıda örnekte bazı etiketler hiç olmazsa uyarı vermeyi engeller
    print(classification_report(df_manuel['label'], tahminler, target_names=ETIKETLER, digits=3, zero_division=0))

manuel_test_et(ornek_veriler)


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Tahminler alınıyor...


AttributeError: 'tuple' object has no attribute 'argmax'

## 8. Tüm pencereleri skorla (~14 bin)

In [ ]:
from tqdm.auto import tqdm

tum = pd.read_csv(f'{KLASOR}/pencereler.csv', encoding='utf-8-sig')
model.eval()
olasiliklar = []
BATCH = 64
for i in tqdm(range(0, len(tum), BATCH)):
    parca = tum['pencere'].iloc[i:i + BATCH].tolist()
    girdi = tokenizer(parca, truncation=True, max_length=MAX_UZUNLUK, padding=True, return_tensors='pt').to(model.device)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        olasiliklar.append(torch.softmax(model(**girdi).logits.float(), dim=-1).cpu().numpy())

P = np.concatenate(olasiliklar)
for j, e in enumerate(ETIKETLER):
    tum[f'p_{e}'] = P[:, j]
tum['tahmin'] = [ETIKETLER[k] for k in P.argmax(1)]
# Makaledeki net skor: P(iyimser) - P(kötümser), [-1, 1]
tum['net_skor'] = tum['p_iyimser'] - tum['p_kotumser']
tum.to_csv(f'{KLASOR}/pencereler_skorlu.csv', index=False, encoding='utf-8-sig')
tum['tahmin'].value_counts()

  0%|          | 0/220 [00:00<?, ?it/s]

,count
tahmin,
kotumser,6004
notr,5173
iyimser,2869


## 9. Kanal-ay iyimserlik endeksi

Makaledeki gibi: önce transkript (dosya) bazında ortalama, sonra kanal-ay bazında ortalama; 5'ten az transkriptli hücreler boş bırakılır.

In [ ]:
transkript = tum.groupby(['kanal', 'grup', 'ay', 'dosya'], as_index=False)['net_skor'].mean()
endeks = transkript.groupby(['kanal', 'grup', 'ay']).agg(net_skor=('net_skor', 'mean'), transkript_sayisi=('dosya', 'count')).reset_index()
endeks.loc[endeks['transkript_sayisi'] < 5, 'net_skor'] = np.nan
endeks.to_csv(f'{KLASOR}/kanal_ay_endeksi.csv', index=False, encoding='utf-8-sig')

import matplotlib.pyplot as plt
RENK = {'hukumete_yakin': 'tab:orange', 'notr': 'tab:blue', 'muhalif': 'black'}
fig, ax = plt.subplots(figsize=(11, 5))
for (kanal, grup), d in endeks.groupby(['kanal', 'grup']):
    ax.plot(d['ay'], d['net_skor'], label=kanal, color=RENK[grup], alpha=0.8)
ax.axhline(0, color='grey', lw=0.5)
ax.set_ylabel('Net iyimserlik  P(iyimser) - P(kötümser)')
ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

NameError: name 'tum' is not defined